In [ ]:
!pip install -q pandas requests openpyxl tqdm python-levenshtein


In [ ]:
import getpass
TMDB_KEY = getpass.getpass("TMDb API key (v3): ")
OMDB_KEY = getpass.getpass("OMDb API key (opcional, enter para saltar): ").strip() or None

TMDb API key (v3): ··········
OMDb API key (opcional, enter para saltar): ··········


In [ ]:
from google.colab import files
uploaded = files.upload()          # escolhe o teu .xlsx
FNAME = list(uploaded.keys())[0]   # usamos a 1ª sheet por defeito
FNAME

Saving DATA SET SEM DAYS OF THE WEEK .xlsx to DATA SET SEM DAYS OF THE WEEK  (1).xlsx


'DATA SET SEM DAYS OF THE WEEK  (1).xlsx'

In [ ]:
import pandas as pd
import requests, time, re
from tqdm import tqdm
from Levenshtein import ratio as lev_ratio

# ========================= Configs =========================
MODE = "blockbuster"        # afinado para cinema comercial
SHEET_NAME = None           # None => 1ª sheet
SLEEP = 0.25
SIM_THRESHOLD = 0.60
ALLOW_OMDB_TITLE_FALLBACK = True   # usar OMDb para descobrir imdbID se TMDb falhar

# ====================== Helpers HTTP (retry) ======================
def http_get(url, params=None, retries=3, timeout=20, headers=None):
    back = 0.6
    for _ in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout, headers=headers)
            if r.status_code in (429,500,502,503,504):
                time.sleep(back); back *= 2; continue
            return r
        except requests.RequestException:
            time.sleep(back); back *= 2
    return None

# ====================== Normalização de título ======================
_clean_re = re.compile(r"[^a-z0-9 ]+")
def norm_title(s: str) -> str:
    s = (s or "").lower().strip()
    s = re.sub(r"\s*\(\d{4}\)$", "", s)
    s = _clean_re.sub(" ", s)
    s = re.sub(r"\s+"," ", s)
    return s.strip()

# ========================= TMDb: search & rank =========================
def tmdb_search_candidates(title, year):
    norm = norm_title(title)
    def q(query, y):
        p = {"api_key": TMDB_KEY, "query": query, "include_adult":"false"}
        if y is not None: p["year"] = int(y)
        r = http_get("https://api.themoviedb.org/3/search/movie", p)
        if not r or r.status_code >= 400: return []
        return r.json().get("results", []) or []
    years = [year] if year is not None else [None]
    if year is not None: years += [year-1, year+1]
    queries = {title, norm}
    cands, seen = [], set()
    for y in years:
        for qstr in queries:
            for it in q(qstr, y):
                k = it.get("id")
                if k not in seen:
                    seen.add(k); cands.append(it)
    return cands

def composite_score(item, want_title, want_year):
    vc = item.get("vote_count", 0) or 0
    pop = item.get("popularity", 0.0) or 0.0
    rd  = item.get("release_date") or ""
    yr  = int(rd[:4]) if len(rd)>=4 and rd[:4].isdigit() else None
    y_pen = abs(yr - want_year) if (want_year and yr) else 0
    sim = lev_ratio(norm_title(item.get("title","")), norm_title(want_title))
    is_doc = 99 in (item.get("genre_ids") or [])
    score = (vc*12) + (pop*2) + (sim*8) - (y_pen*2) - (20 if is_doc else 0)
    return score, sim

def pick_best_tmdb(title, year):
    cands = tmdb_search_candidates(title, year)
    if not cands: return None
    non_docs = [x for x in cands if 99 not in (x.get("genre_ids") or [])]
    pool = non_docs if non_docs else cands
    pool.sort(key=lambda x: composite_score(x, title, year), reverse=True)
    top = pool[0]
    _, sim = composite_score(top, title, year)
    return top if sim >= SIM_THRESHOLD else None

# ========================= TMDb: details & collections =========================
def tmdb_movie_core(tmdb_id):
    """
    Detalhes core numa só estrutura:
    - director (nome,id), lead actor (nome,id), imdb_id, runtime, spoken_languages,
      production_companies (para país), collection info.
    """
    out = {
        "director_name": None, "director_id": None,
        "actor_name": None, "actor_id": None,
        "imdb_id": None, "runtime": None,
        "spoken_languages": None,
        "production_companies": [],
        "collection_id": None, "collection_name": None
    }

    # credits (director / cast)
    r = http_get(f"https://api.themoviedb.org/3/movie/{tmdb_id}/credits", {"api_key": TMDB_KEY})
    if r and r.status_code < 400:
        d = r.json()
        for p in d.get("crew", []):
            if p.get("job") == "Director":
                out["director_name"] = p.get("name"); out["director_id"] = p.get("id"); break
        cast = d.get("cast", [])
        if cast:
            main = sorted(cast, key=lambda x: x.get("order",9999))[0]
            out["actor_name"] = main.get("name"); out["actor_id"] = main.get("id")

    # details (runtime, spoken_languages, production_companies, collection)
    r = http_get(f"https://api.themoviedb.org/3/movie/{tmdb_id}", {"api_key": TMDB_KEY})
    if r and r.status_code < 400:
        j = r.json()
        out["runtime"] = j.get("runtime")
        out["spoken_languages"] = j.get("spoken_languages") or []
        out["production_companies"] = j.get("production_companies") or []
        col = j.get("belongs_to_collection")
        if col:
            out["collection_id"] = col.get("id")
            out["collection_name"] = col.get("name")

    # external ids (imdb)
    r = http_get(f"https://api.themoviedb.org/3/movie/{tmdb_id}/external_ids", {"api_key": TMDB_KEY})
    if r and r.status_code < 400:
        out["imdb_id"] = r.json().get("imdb_id")

    return out

def tmdb_collection_position(collection_id, tmdb_id):
    if not collection_id: return None, None
    r = http_get(f"https://api.themoviedb.org/3/collection/{collection_id}", {"api_key": TMDB_KEY})
    if not r or r.status_code >= 400: return None, None
    parts = r.json().get("parts", []) or []
    def part_key(p):
        rd = p.get("release_date") or ""
        return rd if rd else "9999-99-99"
    parts_sorted = sorted(parts, key=part_key)
    ids = [p.get("id") for p in parts_sorted]
    if tmdb_id in ids:
        return ids.index(tmdb_id) + 1, len(ids)
    return None, len(ids)

# ========================= TMDb: persons (birth year) =========================
person_cache = {}
def tmdb_person_birth_year(person_id):
    if not person_id: return None
    if person_id in person_cache: return person_cache[person_id]
    r = http_get(f"https://api.themoviedb.org/3/person/{person_id}", {"api_key": TMDB_KEY})
    if not r or r.status_code >= 400:
        person_cache[person_id] = None; return None
    j = r.json()
    dob = j.get("birthday")
    by = int(dob[:4]) if dob and len(dob)>=4 and dob[:4].isdigit() else None
    person_cache[person_id] = by
    time.sleep(SLEEP)
    return by

# ========================= OMDb: fallback pontual =========================
def omdb_runtime_and_languages(imdb_id):
    """Retorna (runtime_min, num_languages) a partir do OMDb, se existir."""
    if not imdb_id or not OMDB_KEY: return None, None
    r = http_get("https://www.omdbapi.com/", {"i": imdb_id, "apikey": OMDB_KEY})
    if not r or r.status_code >= 400: return None, None
    d = r.json()
    if d.get("Response") != "True": return None, None
    # runtime
    rt = d.get("Runtime")
    runtime = None
    if rt and isinstance(rt,str) and "min" in rt:
        try: runtime = int(rt.split(" ")[0])
        except: runtime = None
    # languages
    lang = d.get("Language")
    num_langs = len([x.strip() for x in lang.split(",") if x.strip()]) if lang else None
    return runtime, num_langs

# ========================= Wikidata: founded year da company =========================
studio_cache = {}
def wikidata_founded_year(company_name):
    """P571 (inception) via Wikidata. Cache em memória por sessão."""
    if not company_name: return None
    name = company_name.strip()
    if name in studio_cache: return studio_cache[name]

    s = http_get("https://www.wikidata.org/w/api.php",
                 params={"action":"wbsearchentities","search":name,"language":"en","format":"json","type":"item"})
    if not s or s.status_code>=400:
        studio_cache[name] = None; return None
    hits = s.json().get("search", [])
    if not hits:
        studio_cache[name] = None; return None
    qid = hits[0].get("id")

    ent = http_get(f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json")
    if not ent or ent.status_code>=400:
        studio_cache[name] = None; return None
    claims = (ent.json().get("entities", {}).get(qid, {}).get("claims", {}))
    inception = claims.get("P571", [])
    year = None
    if inception:
        mainsn = inception[0].get("mainsnak", {})
        datav = mainsn.get("datavalue", {})
        v = datav.get("value", {})
        time_str = v.get("time")  # "+1923-04-04T00:00:00Z"
        if time_str and len(time_str)>=5 and time_str[1:5].isdigit():
            year = int(time_str[1:5])
    studio_cache[name] = year
    time.sleep(SLEEP)
    return year

# ========================= Pick producer principal =========================
def tmdb_main_producer_id(tmdb_id):
    """Tenta obter o primeiro 'Producer' no crew; fallback: 'Executive Producer'."""
    r = http_get(f"https://api.themoviedb.org/3/movie/{tmdb_id}/credits", {"api_key": TMDB_KEY})
    if not r or r.status_code >= 400: return None
    crew = (r.json().get("crew") or [])
    # Prioridade 'Producer'
    for p in crew:
        if (p.get("job") or "").lower() == "producer":
            return p.get("id")
    # Fallback 'Executive Producer'
    for p in crew:
        if (p.get("job") or "").lower() == "executive producer":
            return p.get("id")
    return None

# ========================= Enrichment por linha =========================
tmdb_row_cache = {}   # (title_norm, year) -> (tmdb_id, details_dict)

def get_row_year(row):
    """Não altera o df. Usa Year se existir; senão tenta derivar localmente de Date."""
    y = None
    if "Year" in row and pd.notna(row["Year"]):
        try: y = int(row["Year"])
        except: y = None
    if y is None and "Date" in row and pd.notna(row["Date"]):
        try:
            y = pd.to_datetime(row["Date"], errors="coerce").year
            if pd.isna(y): y = None
        except: y = None
    return int(y) if y is not None else None

def enrich_row(title, year):
    key = (norm_title(title), year)
    if key in tmdb_row_cache:
        tmdb_id, det = tmdb_row_cache[key]
    else:
        pick = pick_best_tmdb(title, year)
        if not pick and ALLOW_OMDB_TITLE_FALLBACK:
            # Sem TMDb match → tentar descobrir o imdbID e ainda assim obter runtime/lang por OMDb
            det = {"imdb_id": None, "runtime": None, "spoken_languages": [], "production_companies": [], "collection_id": None, "collection_name": None,
                   "director_id": None, "actor_id": None}
            det["imdb_id"] = None  # só descobrimos se precisares; por agora mantemos None
            tmdb_id = None
        elif not pick:
            return [None]*10
        else:
            tmdb_id = pick.get("id")
            det = tmdb_movie_core(tmdb_id)
        tmdb_row_cache[key] = (tmdb_id, det)
        time.sleep(SLEEP)

    # Runtime (TMDb → fallback OMDb)
    runtime = det.get("runtime")
    num_langs = len(det.get("spoken_languages") or [])
    imdb_id = det.get("imdb_id")
    if (runtime is None or num_langs in (None, 0)) and imdb_id:
        rt2, nl2 = omdb_runtime_and_languages(imdb_id)
        if runtime is None: runtime = rt2
        if not num_langs: num_langs = nl2

    # Company (principal) → country + founded year
    companies = det.get("production_companies") or []
    main_company = companies[0] if companies else {}
    company_name = main_company.get("name")
    company_country = main_company.get("origin_country")  # código ISO-3166-1 alpha-2 (ex: US, GB)
    company_founded = wikidata_founded_year(company_name) if company_name else None

    # Sequel/Prequel
    pos, tot = None, None
    is_seq = is_pre = None
    if det.get("collection_id"):
        pos, tot = tmdb_collection_position(det["collection_id"], tmdb_row_cache[key][0])
        is_seq = (pos is not None and pos > 1)
        is_pre = (pos == 1 and (tot or 0) > 1)
    else:
        is_seq = False; is_pre = False

    # Birth years (lead actor + main producer)
    actor_by = tmdb_person_birth_year(det.get("actor_id"))
    prod_id = tmdb_main_producer_id(tmdb_row_cache[key][0]) if tmdb_row_cache[key][0] else None
    producer_by = tmdb_person_birth_year(prod_id)

    return [
        runtime,
        company_founded,
        company_country,
        "Yes" if is_seq else "No",
        "Yes" if is_pre else "No",
        det.get("collection_name"),
        pos, tot,
        actor_by,
        producer_by,
        num_langs
    ]

# ========================= Pipeline =========================
# Ler Excel (NÃO alteramos nenhuma coluna existente)
xls = pd.ExcelFile(FNAME)
if SHEET_NAME is None:
    SHEET_NAME = xls.sheet_names[0]
df = pd.read_excel(FNAME, sheet_name=SHEET_NAME)

# Criar APENAS as colunas novas (se não existirem)
new_cols = [
    "Runtime (min)",
    "Production Company Founded Year",
    "Production Company Country",
    "Is Sequel",
    "Is Prequel",
    "Franchise Name",
    "Franchise Position",
    "Franchise Total",
    "Lead Actor Birth Year",
    "Lead Producer Birth Year",
    "Num Languages"
]
for c in new_cols:
    if c not in df.columns:
        df[c] = None
df[new_cols] = df[new_cols].astype("object")

# Loop (tudo read-only nas colunas originais)
for i in tqdm(range(len(df))):
    title = str(df.loc[i,"Title"]).strip() if "Title" in df.columns else None
    if not title:
        df.loc[i, new_cols] = [None]*len(new_cols); continue
    year = get_row_year(df.loc[i])  # inferido localmente; NÃO é escrito no df
    res = enrich_row(title, year)
    df.loc[i, new_cols] = res

OUTFILE = f"enriched_{FNAME}"
df.to_excel(OUTFILE, index=False)
print("✅ Done →", OUTFILE)

100%|██████████| 34567/34567 [53:11<00:00, 10.83it/s]


✅ Done → enriched_DATA SET SEM DAYS OF THE WEEK  (1).xlsx


In [ ]:
from google.colab import files
files.download(OUTFILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>